In [8]:
import re
import pandas as pd

In [9]:
import pandas as pd
import re

# Đọc dữ liệu
path = "D:\\Career\\Research Paper\\MAPR 2026\\ABSA_ITViec_Reviews\\data\\done\\data_final_sorted.csv"
df = pd.read_csv(path)

# PATTERN TỐI GIẢN & TRIỆT ĐỂ:
# Chỉ cần tìm thấy ký tự '*' là khớp (chúng ta dùng [^ ]* để bao quát nếu cần, 
# nhưng đơn giản nhất cho yêu cầu của bạn là tìm sự tồn tại của '*')
pattern = r'\*' 

# Thực hiện lọc
# na=False để tránh lỗi nếu có dòng bị rỗng (NaN)
df_cleaned = df[~df['review_content'].str.contains(pattern, na=False, regex=True)]

# Hiển thị kết quả kiểm tra
print(f"Số lượng dòng ban đầu: {len(df)}")
print(f"Số lượng dòng sau khi lọc: {len(df_cleaned)}")

# Xuất ra file mới để kiểm tra thủ công
# df_cleaned.to_csv("data_cleaned_no_asterisk.csv", index=False)

Số lượng dòng ban đầu: 19759
Số lượng dòng sau khi lọc: 14846


In [13]:
import re
import pandas as pd

class cleaned_text:
    def __init__(self):
        self.abbreviation_dict = {
            r'\bpv\b|\bpvan\b': 'phỏng vấn',
            r'\bmt\b': 'môi trường',
            r'\bmkt\b': 'marketing',
            r'\bk\b|\bko\b|\bkh\b': 'không',
            r'\bmn\b': 'mọi người',
            r'\bcty\b': 'công ty',
            r'\bsv\b': 'sinh viên',
            r'\bvl\b|\bVL\b': 'vl',
            r'\btts\b': 'thực tập sinh',
            r'\bot\b|\bOT\b|\bovertime\b': 'OT',
            r'\bntn\b': 'như thế nào',
            r'\bae\b|\banhem\b': 'anh em',
            r'\bj\b': 'gì',
            r'\bđc\b': 'được',
            r'\bĐH\b|\bđh\b': 'đại học',
            r'\bbn\b|\bnhieu\b': 'bao nhiêu',
            r'\bnv\b': 'nhân viên',
            r'\bqq\b': 'quần què',
            r'\bah\b': 'ạ',
            r'\bh\b|\bhiem\b': 'bảo hiểm',
        }
        
    def process(self, text):
        import pandas as pd
        # Handle NaN and non-string values
        if pd.isna(text) or not isinstance(text, str):
            return ""
        
        # 1. Chuyển về chữ thường
        text = text.lower()

        # 2. FIX: Thêm khoảng trắng sau dấu câu nếu bị viết dính (Ví dụ: "ạ.Mang" -> "ạ. Mang")
        # Sử dụng Look-ahead để tìm dấu câu dính với chữ cái mà không có khoảng trắng
        text = re.sub(r'([.,!?])(?=[a-zA-Z\u00C0-\u1EF9])', r'\1 ', text)
        text = re.sub(r'\b(v|k|h[aie])\1{1,}\b', ' ', text)

        # 3. FIX: Xóa các dấu gạch đầu dòng "-" 
        # Chúng ta xóa dấu gạch đơn độc, không xóa dấu gạch nối trong từ ghép (nếu có)
        text = re.sub(r'(^|\s)-\s*', ' ', text)

        # Tìm dấu chấm xuất hiện từ 2 lần trở lên và thay bằng 1 dấu chấm
        text = re.sub(r'\.{2,}', '.', text)
        text = re.sub(r'\!+', '!', text)
        # \?+ tìm 1 hoặc nhiều dấu ?, thay bằng ?
        text = re.sub(r'\?+', '?', text)

        # 4. Xử lý icon/emoticons (Thay bằng khoảng trắng)
        text = re.sub(r'[:=;x][\-\^]?[v3ddppsso038]', '', text)
        text = re.sub(r'\^{2,}', '', text)
        text = re.sub(r'[:=\-][\)\(\]\[]{1,}', '', text)

        # 5. Xử lý triệt để tiếng cười kkk, kk, k k k
        text = re.sub(r'\b(k\s*){2,}\b', ' ', text)
        text = re.sub(r'\b(ok\s*){2,}\b', 'ok ', text)
        text = re.sub(r'\b(ha\s*){2,}\b|\b(hi\s*){2,}\b|\b(he\s*){2,}\b', ' ', text)

        # 6. Chuẩn hóa từ lặp (koooooo -> ko)
        text = re.sub(r'([a-z\u00C0-\u1EF9])\1{2,}', r'\1', text)

        # 7. Thay thế từ viết tắt
        for pattern, replacement in self.abbreviation_dict.items():
            text = re.sub(pattern, replacement, text)

        # 8. Xóa ký tự đặc biệt còn sót lại (Chỉ giữ chữ, số, và dấu câu cơ bản)
        text = re.sub(r'[^\s\w\d_.,!?\u00C0-\u1EF9]', ' ', text)

        # 9. CHUẨN HÓA KHOẢNG TRẮNG (Xử lý dư thừa sau khi xóa các ký tự trên)
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text.strip()

# --- Test kiểm chứng các lỗi bạn vừa nêu ---
cleaner = cleaned_text()
test_cases = [
    "Công ty quá tệ!!!!! :vvv Lương thấp quá trời luôn...... =))))) HR thì lồi lõm???",
    "Công cty làm ăn bậy bạ và đâm thọt ạ.Mang danh..", # Lỗi dính dấu chấm
    " - Trước khi đi phỏng vấn mình cũng lên...",      # Lỗi gạch đầu dòng
    "-vấn đề lương bổng 🥬.vấn đề sếp:v",                  # Kết hợp nhiều lỗi
]

for t in test_cases:
    print(f"Gốc: {t}")
    print(f"Sạch: {cleaner.process(t)}")
    print("-" * 30)

Gốc: Công ty quá tệ!!!!! :vvv Lương thấp quá trời luôn...... =))))) HR thì lồi lõm???
Sạch: công ty quá tệ! lương thấp quá trời luôn. hr thì lồi lõm?
------------------------------
Gốc: Công cty làm ăn bậy bạ và đâm thọt ạ.Mang danh..
Sạch: công công ty làm ăn bậy bạ và đâm thọt ạ. mang danh.
------------------------------
Gốc:  - Trước khi đi phỏng vấn mình cũng lên...
Sạch: trước khi đi phỏng vấn mình cũng lên.
------------------------------
Gốc: -vấn đề lương bổng 🥬.vấn đề sếp:v
Sạch: vấn đề lương bổng . vấn đề sếp
------------------------------


In [14]:
df_cleaned["review_content_cleaned"] = df_cleaned["review_content"].apply(cleaner.process)

C:\Users\X1 PRO\AppData\Local\Temp\ipykernel_6032\3038529099.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned["review_content_cleaned"] = df_cleaned["review_content"].apply(cleaner.process)


In [17]:
df_cleaned.to_csv("D:\\Career\\Research Paper\\MAPR 2026\\ABSA_ITViec_Reviews\\data\\done\\data_final_sorted_cleaned.csv", index=False, encoding='utf-8-sig')